# 14. Memory-Aware Chatbot with Mem0 + Vector DB
**Industry:** Tourism

Build a chatbot with hybrid memory using Mem0 and dynamically retrieve preferences.

In [ ]:
!pip install mem0ai langchain-google-genai

In [ ]:
import os
from mem0 import Memory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Mem0 setup (uses default local SQLite/Chroma if no cloud API keys are provided)
os.environ["OPENAI_API_KEY"] = "mock" # Mock key to prevent mem0 from complaining if using local embeddings
memory = Memory()

# We will use Gemini for the conversational generation
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

user_id = "traveler_99"

def chat(message: str):
    # 1. Retrieve relevant memories
    previous_memories = memory.search(message, user_id=user_id)
    context = "\n".join([m["memory"] for m in previous_memories]) if previous_memories else "None"
    
    # 2. Generate response with context
    prompt = f"You are a travel concierge. \nUser Preferences/History: {context}\n\nUser: {message}"
    response = llm.invoke([HumanMessage(content=prompt)])
    
    # 3. Store the new message in memory for future extraction
    memory.add(message, user_id=user_id)
    
    return response.content

print("--- Session 1 ---")
print("User: I am planning a trip to Japan. I love window seats and am strictly vegetarian.")
print("Bot:", chat("I am planning a trip to Japan. I love window seats and am strictly vegetarian."))

print("\n--- Session 2 (Later) ---")
print("User: Book me a flight to Tokyo and suggest a restaurant.")
print("Bot:", chat("Book me a flight to Tokyo and suggest a restaurant."))